In [ ]:
# @title Imports! Impor(s)tant!
import numpy as np
! pip install numpy-financial
import numpy_financial as npf
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# @title Dictionary Defenitions (press to reset)
substanceDict = {}
costDict = {}
systemVariables = {}

In [ ]:
# @title Global variables
# @markdown **Global Variables**
# @markdown  -
# @markdown inflation and interest rate
inflation = 0.035 # @param {"type":"slider", min:0, max:0.2, step:0.005}
interest = 0.06 # @param {"type":"slider","min":0,  max:0.2, step:0.005}
# @markdown Beta power sector (negative = volitile, positive = stable)
beta = 0.79 # @param {"type":"slider", min:-1, max:1, step:0.01}
# @markdown Equaty
equityRiskPremium = 0.055 # @param {"type":"slider", min:0, max:5, step:0.01}
equityRatio = 1 #@param {"type":"slider", min:-1, max:1, step:0.1}
# @markdown Project variables
lifetime = 20 #@param (type:"number")
# @markdown Corporate tax
corpTax = 0.258 #@param {type:"slider", min:0, max:1,step:0.01}

systemVariables["inflation"] = inflation
systemVariables["interest"] = interest
systemVariables["beta"] = beta
systemVariables["equityRiskPremium"] = equityRiskPremium
systemVariables["equityRatio"] = equityRatio
systemVariables["lifetime"] = lifetime
systemVariables["corpTax"] = corpTax


In [ ]:
# @title Adding subtances
# @markdown Variables

substance = "" # @param {type:"string"}
# @markdown Flow in kg/year
input = None # @param {type:"number"}
output = None # @param {type:"number"}
# @markdown type of substance
substanceType = "unused" # @param ["starting material" ,"unused", "product"]
# @markdown Cost per kg
cost = None # @param {type:"number"}

substanceDict[substance] = {"input": input, "output": output, "net": (input-output), "type":substanceType, "cost": cost}

In [ ]:
# @title Adding Non-Material costs
# @markdown What is the cost?

source = "" # @param {type:"string"}

# @markdown Type of Cost and amount in Euro
costType = "once" # @param ["once","yearly","yearly per substance"]
costs = None # @param {type:"number"}
perSubstanceKey = '' # @param {type:"string"}

# @markdown Fixed or Variable
costCatagory = "fixed" # @param ["fixed","variable", "other"]

costDict[source] = {"type": costType, "costs": costs, "substanceKey":perSubstanceKey, "category":costCatagory}


In [ ]:
print(substanceDict)
print(costDict)
print(systemVariables)

In [ ]:
# @title Default Settings (Brewery example)
def addDefaultBreweryTest():
  # adding substances
  substanceDict["beer"] = {'input': 0, 'output': 2176200, 'net': -2176200, 'type': 'product', 'cost': 1.22876}
  substanceDict["water"] = {'input': 2067390, 'output': 0, 'net': 2067390, 'type': 'starting material', 'cost': 0.00145}
  substanceDict["barley"] = {'input': 435240, 'output': 0, 'net': 435240, 'type': 'starting material', 'cost': 2.08}
  substanceDict["hop"] = {'input': 4352, 'output': 0, 'net': 4352, 'type': 'starting material', 'cost': 36.344}
  substanceDict["yeast"] = {'input': 1088, 'output': 0, 'net': 1088, 'type': 'starting material', 'cost': 140.217}

  #adding costs (initial costs)
  costDict["basic install costs watersupply"] = {'type': 'once', 'costs': 1708,'substanceKey':'', 'category': 'fixed'}
  costDict["machine costs"] = {'type': 'once', 'costs': 201078,'substanceKey':'', 'category': 'fixed'}
  costDict["initial location costs"] = {'type': 'once', 'costs': 41760,'substanceKey':'', 'category': 'fixed'}
  #adding costs (yearly)
  costDict["cleaner"] = {'type': 'yearly', 'costs': 5184,'substanceKey':'', 'category': 'fixed'}
  costDict["hiring Location"] = {'type': 'yearly', 'costs': 41760,'substanceKey':'', 'category': 'fixed'}
  costDict["total energy price"] = {'type': 'yearly', 'costs': 185660.87,'substanceKey':'', 'category': 'variable'}
  #adding substancedependent costs
  costDict["bottles"] = {'type': 'yearly per substance', 'costs': 0.55,'substanceKey':'beer', 'category': 'variable'}
  costDict["tax on water"] = {'type': 'yearly per substance', 'costs': 0.00043,'substanceKey':'water','category': 'variable'}

  #adding systemvariables
  systemVariables['inflation'] = 0.035
  systemVariables['interest'] = 0.06
  systemVariables['beta'] = 0.79
  systemVariables['equityRiskPremium'] = 0.055
  systemVariables['equityRatio'] = 1
  systemVariables['lifetime'] = 20
  systemVariables['corpTax'] = 0.19


addDefaultBreweryTest()

In [ ]:
# @title Test Creation : with Inflation

class companyToBeAnalysed():
  def __init__(self, substanceDict, costDict, systemVariables):

    #Get the important lists
    self.substanceDict = substanceDict
    self.costDict = costDict

    #unpack some of system variables
    self.interest = systemVariables["interest"]
    self.inflation = systemVariables["inflation"]
    self.beta = systemVariables["beta"]
    self.equityRiskPremium = systemVariables["equityRiskPremium"]
    self.equityRatio = systemVariables["equityRatio"]
    self.corporateTax = systemVariables["corpTax"]
    self.lifetime = systemVariables["lifetime"]

    #Calculate the RFIR and WACC because im sure we're going to need it somewhere
    self.RFIR = ((1 + self.interest)/(1 + self.inflation)) - 1
    self.WACC = ((self.RFIR) + (self.beta * self.equityRiskPremium))

    self.runSim()
    #self.printStuff()
    self.makeGraph()


  def runSim(self):
    self.fixedCostContainter = []
    self.variableCostContainer = []
    self.revenueContainer = []
    self.operatingIncomeContainer = []
    self.netIncomeContainer = []
    self.discountedCashFlowContainer = []
    self.CumDCFContainer = []

    def calculateBreakeven(self):
      for i in range(len(self.CumDCFContainer)):
        if self.CumDCFContainer[i] <= 0:
          return ((i-1) + ((-self.CumDCFContainer[i-1]) / (self.CumDCFContainer[i]-self.CumDCFContainer[i-1])))
      return None

    def calculateInitialCosts(self):
      cost = 0
      for key in self.costDict.keys():
        if self.costDict[key]["type"] == "once":
          cost += self.costDict[key]["costs"]
      return cost

    def calculateYearlyFixed(self):
      cost = 0

      #Add in other costs
      for key in self.costDict.keys():
        if self.costDict[key]["type"] == "yearly" and self.costDict[key]["category"] == "fixed":
          cost += self.costDict[key]["costs"]
      return cost

    def calculateYearlyCosts(self):
      cost = 0
      for key in self.substanceDict.keys():
        if self.substanceDict[key]["net"] > 0:
          cost += self.substanceDict[key]["net"] * self.substanceDict[key]["cost"]

      materialcost = cost

      #Add The yearly variable costs and costs of perSubstance used costs
      for key in self.costDict.keys():
        if self.costDict[key]["type"] == "yearly" and self.costDict[key]["category"] == "variable":
          cost += self.costDict[key]["costs"]
        if self.costDict[key]["type"] == "yearly per substance" and self.costDict[key]["category"] == "variable":
          cost += self.costDict[key]["costs"] * abs(self.substanceDict[self.costDict[key]["substanceKey"]]["net"])

      return cost

    #Basically the same function and i dont like it. Probs could be optimised
    def calculateYearlyRev(self):
      totYearlyrev = 0
      for key in self.substanceDict.keys():
        if self.substanceDict[key]["type"] == "product":
          totYearlyrev += -self.substanceDict[key]["net"] * (self.substanceDict[key]["cost"]) # * 1000 <- dono why this was in there but im not sure it helps...
      return totYearlyrev

    def calculateInflation(self, year):
      return 1
      #return ((1 + self.inflation) ** year)


    #A function to set up the initial state. Which is substantially different from the following years
    def setupInitialState(self):
      self.fixedCostContainter.append(-calculateInitialCosts(self)) # Initial cost per capacity * capacity
      self.variableCostContainer.append(0) # 0 because no variable costs yet, factory has not yet been operating
      self.revenueContainer.append(0) # 0 because nothing has been sold yet
      self.operatingIncomeContainer.append(self.fixedCostContainter[-1] + self.variableCostContainer[-1] + self.revenueContainer[-1]) # sum of the above 3 variable
      self.netIncomeContainer.append(self.variableCostContainer[-1] + self.revenueContainer[-1] + self.operatingIncomeContainer[-1]) # sum of the above 3 variables
      self.discountedCashFlowContainer.append(self.netIncomeContainer[-1]) #same as net income
      self.CumDCFContainer.append(self.netIncomeContainer[-1]) # same as net income
      return

    #A function to set up the following years making a nice
    def runYearly(self, year):
      self.fixedCostContainter.append(-calculateYearlyFixed(self) * calculateInflation(self,year)) #same calculation as year 0
      self.variableCostContainer.append(-calculateYearlyCosts(self) * calculateInflation(self,year)) # sum of all costs per product * the amount of product made
      self.revenueContainer.append(calculateYearlyRev(self) * calculateInflation(self,year)) #Selling price * the amount of product made
      self.operatingIncomeContainer.append(self.fixedCostContainter[-1] + self.variableCostContainer[-1] + self.revenueContainer[-1]) # sum of the above 3 variable
      self.netIncomeContainer.append((1-self.corporateTax) * self.operatingIncomeContainer[-1]) # epic
      self.discountedCashFlowContainer.append(npf.pv(rate = self.WACC, nper = year, pmt = 0, fv = -self.netIncomeContainer[-1])) # calculates the discounted cashflow using the numpy method for Present Value
      self.CumDCFContainer.append(self.CumDCFContainer[-1] + self.discountedCashFlowContainer[-1]) # previous CDCF + Discounted cashflow
      return

  #Acutal logic
    setupInitialState(self)
    for i in range(1,self.lifetime+1):
      runYearly(self, i)

    self.IRR = npf.irr(self.netIncomeContainer)
    self.NPV = npf.npv(self.WACC, self.netIncomeContainer)

    self.breakeven = calculateBreakeven(self)

    print(f"The Internal Rate of Return is {round(self.IRR * 100,2)}%")
    print(f"The NetPresentValue is ${round(self.NPV,2)}")
    print(f"the Breakevenpoint is after {round(self.breakeven,2)}years")

    return

  def makeGraph(self):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.fixedCostContainter, name="Fixed Costs"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.variableCostContainer, name="Variable Costs"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.revenueContainer, name="Revenue"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.operatingIncomeContainer, name="Operating Income"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.netIncomeContainer, name="Net Income"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.discountedCashFlowContainer, name="Discounted Cash Flow"))
    fig.add_trace(go.Scatter(x=[x for x in range(0,self.lifetime+1)], y=self.CumDCFContainer, name="Cummulative DCF"))

    fig.show()
    return

  def printStuff(self):
    print("Fixed, Variable and Revenue")
    print(self.fixedCostContainter)
    print(self.variableCostContainer)
    print(self.revenueContainer)
    print("Operating Income and Net Income")
    print(self.operatingIncomeContainer)
    print(self.netIncomeContainer)
    print("Discounted Cash Flow")
    print(self.discountedCashFlowContainer)
    print("Cummulative DCF")
    print(self.CumDCFContainer)
    print(f"The Internal Rate of Return is {round(self.IRR * 100,2)}%")
    print(f"The NetPresentValue is ${round(self.NPV,2)}")
    print(f"The breakeven point is {round(self.breakeven,2)}")
    return

  def getData(self):
    TotalDict = {
        "FixedCosts" : self.fixedCostContainter
        ,"VariableCosts" : self.variableCostContainer
        ,"Revenue" : self.revenueContainer
        ,"OperatingIncome" : self.operatingIncomeContainer
        ,"NetIncome" : self.netIncomeContainer
        ,"DiscountedCashFlow" : self.discountedCashFlowContainer
        ,"CummulativeDCF" : self.CumDCFContainer
        ,"InternalRateOfReturn" : self.IRR
        ,"NetPresentValue" : self.NPV
        ,"Breakeven" : self.breakeven
      }
    return TotalDict

  def getnpv(self):
    return self.NPV



In [ ]:
# @title Default Settings (Brewery example)
def addDefaultBreweryTest():
  # adding substances
  #substanceDict["elctricity"] = {'input': , 'output': , 'net': , 'type': 'product', 'cost': 1.22876}

  #adding costs (initial costs)
  # Retrofit cost = Cost of new model - Cost of old model in this case, So : 5.930E+005 - 4.896E+005 = 1.034E+005
  costDict["Retrofitting HEN"] = {'type': 'once', 'costs': 103400,'substanceKey':'', 'category': 'fixed'}

  #adding costs (yearly)
  #lets hope this doesnt break the math
  #costDict["Electricity"] = {'type': 'yearly', 'costs': , 'substanceKey':'', 'category':'variable'}
  costDict['Total Costs'] = {'type': 'yearly', 'costs': -259500, 'substanceKey':'', 'category':'variable'}

  #adding substancedependent costs

  #adding systemvariables
  systemVariables['inflation'] = 0.035
  systemVariables['interest'] = 0.06
  systemVariables['beta'] = 0.79
  systemVariables['equityRiskPremium'] = 0.055
  systemVariables['equityRatio'] = 1
  systemVariables['lifetime'] = 20
  systemVariables['corpTax'] = 0.19

addDefaultBreweryTest()

pog = companyToBeAnalysed(substanceDict, costDict, systemVariables)


The Internal Rate of Return is 203.28%
The NetPresentValue is $2165476.22
the Breakevenpoint is after -2.05years


In [ ]:
# @title Sensitivity Analysis
fig = go.Figure()

#parameters = [('system','inflation'), ('system','interest'), ('system','beta'),
              #('system','equityRiskPremium'), ('substance', 'beer', 'cost'),('substance', 'water', 'cost'),
              #('substance', 'barley', 'cost'),('substance', 'hop', 'cost'),('substance', 'yeast', 'cost'),
              #('cost', 'bottles', 'costs'),('cost', 'cleaner', 'costs'),('cost', 'hiring Location', 'costs'),
              #('cost','total energy price', 'costs')]

parameters = [('system','inflation')]

dictDict = {'substance':substanceDict, 'cost':costDict,'system':systemVariables}
for items in parameters:
  valuelist = []
  npvresults = []
  if len(items) == 2: actualvalue = dictDict[items[0]][items[1]]
  else: actualvalue = dictDict[items[0]][items[1]][items[2]]
  for value in [(actualvalue + actualvalue * (x/100))  for x in range(-20,21)]:
    valuelist.append(value)
    if len(items) == 2: dictDict[items[0]][items[1]] = value
    else: dictDict[items[0]][items[1]][items[2]] = value
    brewery = companyToBeAnalysed(dictDict["substance"], dictDict["cost"], dictDict["system"])
    npvresults.append(brewery.getnpv())
  if len(items) == 2:
    dictDict[items[0]][items[1]] = actualvalue
    fig.add_trace(go.Scatter(x=[f"{x}%" for x in range(-20,21)], y=npvresults, name=items[1]))
  else:
    dictDict[items[0]][items[1]][items[2]] = actualvalue
    fig.add_trace(go.Scatter(x=[f"{x}%" for x in range(-20,21)], y=npvresults, name=f"{items[1]} {items[2]}"))

fig.update_layout(title=dict(text="Sensitivity Analysis"), xaxis=dict(title=dict(text="Percentage change")), yaxis=dict(title=dict(text="NPV")))
fig.show()